# Qualitative Figure + Probability Heatmap

Run this notebook in Google Colab. It creates a fresh PDF version of the qualitative figure with an added `P(ours + prior)` probability heatmap column. It does **not** modify your existing project notebooks.

Expected Drive layout from the project runs:

- `/content/drive/MyDrive/HLCV/checkpoints/{dtd_doctamper.pth,vph_imagenet.pt,swin_imagenet.pt}`
- `/content/drive/MyDrive/HLCV_samira/final_seed42/arms/easyocr.pth`
- `/content/drive/MyDrive/HLCV_samira/ta_design/.../*.pth` for the text-head column
- `/content/drive/MyDrive/HLCV_samira/manifests/train800_val200_test200_seed42/test.json`
- `/content/drive/MyDrive/HLCV_samira/data/doctamper.zip` containing `DocTamperV1-TestingSet`

The output PDF will be saved to Drive and a PNG preview will be displayed below.


In [10]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [11]:
# Paths you may edit if your Drive folders are different.
from pathlib import Path

PROJECT_REPO_URL = 'https://github.com/SamiraAbedini/HLCV-Project.git'
PROJECT_BRANCH = 'main'
DOCTAMPER_REPO_URL = 'https://github.com/qcf-568/DocTamper.git'

PROJECT_DIR = Path('/content/HLCV-Project')
DOCTAMPER_DIR = Path('/content/DocTamper')
DATA_ROOT = Path('/content/doctamper_train_test')

CHECKPOINT_DIR = Path('/content/drive/MyDrive/HLCV/checkpoints')
FINAL_OUT = Path('/content/drive/MyDrive/HLCV_samira/final_seed42')
TA_OUT = Path('/content/drive/MyDrive/HLCV_samira/ta_design')
MANIFEST_DIR = Path('/content/drive/MyDrive/HLCV_samira/manifests/train800_val200_test200_seed42')
DRIVE_ZIP = Path('/content/drive/MyDrive/HLCV_samira/data/doctamper.zip')

OUT_PDF = Path('/content/drive/MyDrive/HLCV_samira/qualitative/qualitative_with_probability_heatmap.pdf')
PRIOR_ARM = 'easyocr'

import os
for _name in [
    'PROJECT_DIR', 'DOCTAMPER_DIR', 'DATA_ROOT', 'CHECKPOINT_DIR',
    'FINAL_OUT', 'TA_OUT', 'MANIFEST_DIR', 'DRIVE_ZIP', 'OUT_PDF', 'PRIOR_ARM'
]:
    os.environ[_name] = str(globals()[_name])


In [12]:
# Clone repos and install dependencies. This can take a few minutes on a fresh Colab runtime.
import os, subprocess, sys

os.chdir('/content')
if not PROJECT_DIR.exists():
    subprocess.run(['git', 'clone', '-q', '-b', PROJECT_BRANCH, PROJECT_REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', PROJECT_BRANCH], check=False)
if not DOCTAMPER_DIR.exists():
    subprocess.run(['git', 'clone', '-q', DOCTAMPER_REPO_URL, str(DOCTAMPER_DIR)], check=True)

!pip -q install --use-deprecated=legacy-resolver lmdb six albumentations timm==0.4.12 segmentation_models_pytorch==0.2.1 easyocr pytesseract kaggle scikit-learn matplotlib opencv-python efficientnet_pytorch==0.7.1
!apt-get -qq install -y tesseract-ocr libjpeg-dev > /dev/null
!pip -q uninstall -y jpegio
!rm -rf /content/jpegio
!git clone -q https://github.com/dwgoon/jpegio.git /content/jpegio
%cd /content/jpegio
!pip -q install .
%cd /content


  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
ERROR: Cannot install efficientnet_pytorch==0.7.1 and segmentation-models-pytorch==0.2.1 because these package versions have conflicting dependencies.
ERROR: ResolutionImpossible: for help visit https://pip.pypa.io/en/latest/topics/dependency-resolution/#dealing-with-dependency-conflicts
/content/jpegio
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
/content


In [13]:
# Unzip the TestingSet if it is not already present in the Colab runtime.
DATA_ROOT.mkdir(parents=True, exist_ok=True)
need = DATA_ROOT / 'DocTamperV1-TestingSet' / 'data.mdb'
if not need.exists():
    assert DRIVE_ZIP.exists(), f'Missing dataset zip: {DRIVE_ZIP}'
    !unzip -o -q "$DRIVE_ZIP" "DocTamperV1-TestingSet/*" -d "$DATA_ROOT"
assert need.exists(), f'Missing TestingSet LMDB after unzip: {need}'
print('TestingSet ready:', need)


TestingSet ready: /content/doctamper_train_test/DocTamperV1-TestingSet/data.mdb


In [18]:
# Write the standalone generator into the Colab clone. This keeps the notebook self-contained.
script_path = PROJECT_DIR / 'scripts' / 'make_qualitative_with_probability_heatmap.py'
script_path.parent.mkdir(parents=True, exist_ok=True)
script_path.write_text('"""Build the qualitative figure with an added tamper-probability heatmap.\n\nThis script is meant to be run in the same Colab/Drive environment used for\n`qualitative_redundancy_colab.ipynb` and `final_results_colab.ipynb`. It does\nnot modify either notebook. It recomputes the baseline and +prior probability\nmaps, selects the same kind of examples used in the presentation, and writes a\npublication-ready PDF.\n\nExample:\n    python scripts/make_qualitative_with_probability_heatmap.py\n"""\n\nfrom __future__ import annotations\n\nimport argparse\nimport os\nfrom pathlib import Path\nimport shutil\nimport sys\n\nimport numpy as np\n\n\nMEAN = np.array([0.485, 0.455, 0.406])\nSTD = np.array([0.229, 0.224, 0.225])\nIMG = 512\n\n\ndef parse_args() -> argparse.Namespace:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--project-dir", default="/content/HLCV-Project")\n    parser.add_argument("--doctamper-dir", default="/content/DocTamper")\n    parser.add_argument("--data-root", default="/content/doctamper_train_test")\n    parser.add_argument(\n        "--manifest-dir",\n        default="/content/drive/MyDrive/HLCV_samira/manifests/train800_val200_test200_seed42",\n    )\n    parser.add_argument("--checkpoint-dir", default="/content/drive/MyDrive/HLCV/checkpoints")\n    parser.add_argument("--final-out", default="/content/drive/MyDrive/HLCV_samira/final_seed42")\n    parser.add_argument("--ta-out", default="/content/drive/MyDrive/HLCV_samira/ta_design")\n    parser.add_argument(\n        "--out",\n        default="/content/drive/MyDrive/HLCV_samira/qualitative/qualitative_with_probability_heatmap.pdf",\n    )\n    parser.add_argument("--prior-arm", default="easyocr")\n    parser.add_argument("--init-checkpoint", default="dtd_doctamper.pth")\n    parser.add_argument("--jpeg-quality", type=int, default=75)\n    parser.add_argument("--threshold", type=float, default=0.5)\n    parser.add_argument("--text-feat-idx", type=int, default=0)\n    parser.add_argument("--ocr-confidence-threshold", type=float, default=30.0)\n    parser.add_argument("--ocr-dilation", type=int, default=2)\n    parser.add_argument("--ocr-languages", default="eng")\n    parser.add_argument("--n-top-diff", type=int, default=4)\n    parser.add_argument("--explicit-indices", default="")\n    parser.add_argument("--scan-limit", type=int, default=0)\n    return parser.parse_args()\n\n\ndef require(path: Path, note: str) -> Path:\n    if not path.exists():\n        raise FileNotFoundError(f"Missing {note}: {path}")\n    return path\n\n\ndef denorm(t):\n    arr = t.permute(1, 2, 0).cpu().numpy() * STD + MEAN\n    return np.clip(arr * 255, 0, 255).astype(np.uint8)\n\n\ndef batch_from_sample(sample):\n    import torch\n\n    batch = {}\n    for key, value in sample.items():\n        if torch.is_tensor(value):\n            batch[key] = value.unsqueeze(0)\n        elif isinstance(value, np.ndarray):\n            batch[key] = torch.from_numpy(value).unsqueeze(0)\n        else:\n            batch[key] = [value]\n    return batch\n\n\ndef copy_if_needed(src: Path, dst: Path) -> None:\n    if not dst.exists():\n        shutil.copy2(src, dst)\n\n\ndef setup_runtime(args: argparse.Namespace):\n    project_dir = require(Path(args.project_dir), "project checkout")\n    doctamper_models = require(Path(args.doctamper_dir) / "models", "DocTamper/models")\n    sys.path.insert(0, str(doctamper_models))\n    sys.path.insert(0, str(project_dir))\n    os.chdir(doctamper_models)\n\n    checkpoint_dir = require(Path(args.checkpoint_dir), "checkpoint directory")\n    copy_if_needed(require(Path(args.doctamper_dir) / "qt_table.pk", "qt_table.pk"), Path("qt_table.pk"))\n    for name in ["vph_imagenet.pt", "swin_imagenet.pt", args.init_checkpoint]:\n        copy_if_needed(require(checkpoint_dir / name, name), Path(name))\n\n\ndef build_components(args: argparse.Namespace):\n    import torch\n    import torch.nn as nn\n    import torch.nn.functional as F\n    from torch.cuda.amp import autocast\n    from torch.utils.data import DataLoader\n\n    if not getattr(torch.load, "_hlcv_weights_only_compat", False):\n        original_torch_load = torch.load\n\n        def torch_load_compat(*load_args, **load_kwargs):\n            load_kwargs.setdefault("weights_only", False)\n            return original_torch_load(*load_args, **load_kwargs)\n\n        torch_load_compat._hlcv_weights_only_compat = True\n        torch.load = torch_load_compat\n\n    doctamper_models = Path(args.doctamper_dir) / "models"\n    if str(doctamper_models) not in sys.path:\n        sys.path.insert(0, str(doctamper_models))\n\n    from dtd import seg_dtd\n    from src.doctamper_dataset import ManifestDocTamperDataset\n    from src.fusion import TextPriorFusion\n    from src.ocr_backends import OCRConfig, create_ocr_backend\n    from src.ocr_cache import OCRDetectionCache\n\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    torch.manual_seed(42)\n    np.random.seed(42)\n\n    data_root = require(Path(args.data_root), "DocTamper data root")\n    manifest = require(Path(args.manifest_dir) / "test.json", "test manifest")\n    test_ds = ManifestDocTamperDataset(data_root, manifest, "qt_table.pk", args.jpeg_quality)\n    test_loader = DataLoader(test_ds, batch_size=1, shuffle=False, num_workers=0)\n\n    def build_model():\n        model = seg_dtd("", 2).to(device)\n        for mod in model.modules():\n            if isinstance(mod, nn.GELU) and not hasattr(mod, "approximate"):\n                mod.approximate = "none"\n        state = torch.load(args.init_checkpoint, map_location="cpu")["state_dict"]\n        model.load_state_dict({k.replace("module.", ""): v for k, v in state.items()}, strict=False)\n        return model\n\n    def forward_dtd(model, batch):\n        return model(batch["image"].to(device), batch["rgb"].to(device), batch["q"].unsqueeze(1).to(device))\n\n    class HeadWithPrior(nn.Module):\n        def __init__(self, head, in_ch):\n            super().__init__()\n            self.fusion = TextPriorFusion(in_ch)\n            self.head = head\n            self.text_mask = None\n\n        def forward(self, feat):\n            return self.head(self.fusion(feat, self.text_mask))\n\n    def wire_prior(model):\n        core = model.model\n        head = core.segmentation_head\n        while hasattr(head, "head"):\n            head = head.head\n        core.segmentation_head = HeadWithPrior(head, head[0].in_channels).to(device)\n        return core\n\n    class TextHead(nn.Module):\n        def __init__(self, in_ch, mid=64):\n            super().__init__()\n            self.net = nn.Sequential(\n                nn.Conv2d(in_ch, mid, 3, padding=1, bias=False),\n                nn.BatchNorm2d(mid),\n                nn.ReLU(True),\n                nn.Conv2d(mid, mid, 3, padding=1, bias=False),\n                nn.BatchNorm2d(mid),\n                nn.ReLU(True),\n                nn.Conv2d(mid, 1, 1),\n            )\n\n        def forward(self, x):\n            return self.net(x)\n\n    languages = tuple(x.strip() for x in args.ocr_languages.split(",") if x.strip())\n    backend_langs = tuple("en" if x == "eng" else x for x in languages) if args.prior_arm == "easyocr" else languages\n    conf = 0.0 if args.prior_arm == "easyocr" and args.ocr_confidence_threshold > 1 else args.ocr_confidence_threshold\n    ocr_cfg = OCRConfig(\n        backend=args.prior_arm,\n        languages=backend_langs,\n        confidence_threshold=conf,\n        dilation=args.ocr_dilation,\n        easyocr_gpu=(device == "cuda"),\n    )\n    ocr_backend = create_ocr_backend(ocr_cfg)\n    ocr_cache = OCRDetectionCache(Path(args.final_out) / "ocr_cache", ocr_cfg)\n\n    @torch.no_grad()\n    def predict(model, idx, use_prior):\n        sample = test_ds[idx]\n        batch = batch_from_sample(sample)\n        if use_prior:\n            image = denorm(sample["image"])\n            det, _ = ocr_cache.get_or_compute(sample["sample_id"], image, ocr_backend)\n            mask = ocr_cache.mask_from_detections(det, image.shape)\n            model.model.segmentation_head.text_mask = torch.from_numpy(mask[None, None]).float().to(device)\n        with autocast(enabled=(device == "cuda")):\n            logits = forward_dtd(model, batch)\n        logits = F.interpolate(logits.float(), size=(IMG, IMG), mode="bilinear", align_corners=False)\n        return torch.softmax(logits, 1)[0, 1].cpu().numpy()\n\n    @torch.no_grad()\n    def capture_row(base_model, text_head, idx):\n        sample = test_ds[idx]\n        batch = batch_from_sample(sample)\n        grabbed = {}\n        hook = base_model.model.vph.register_forward_hook(lambda _m, _i, out: grabbed.update(out=out))\n        with autocast(enabled=(device == "cuda")):\n            _ = forward_dtd(base_model, batch)\n        hook.remove()\n        feats = list(grabbed["out"]) if isinstance(grabbed["out"], (list, tuple)) else [grabbed["out"]]\n        feat = feats[args.text_feat_idx].float()\n        text_prob = None\n        if text_head is not None:\n            text_logits = text_head(feat)\n            text_logits = F.interpolate(text_logits, size=(IMG, IMG), mode="bilinear", align_corners=False)\n            text_prob = torch.sigmoid(text_logits)[0, 0].cpu().numpy()\n        image = denorm(sample["image"])\n        det, _ = ocr_cache.get_or_compute(sample["sample_id"], image, ocr_backend)\n        ocr_mask = ocr_cache.mask_from_detections(det, image.shape).astype(np.float32)\n        gt = sample["label"][0].numpy() > 0\n        return {"image": image, "ocr": ocr_mask, "text": text_prob, "gt": gt, "sample_id": sample["sample_id"]}\n\n    def load_text_head():\n        channels = [96, 192, 384, 768][args.text_feat_idx]\n        for candidate in sorted(Path(args.ta_out).glob("**/*.pth")):\n            try:\n                blob = torch.load(candidate, map_location="cpu")\n            except Exception:\n                continue\n            if isinstance(blob, dict) and "text_head" in blob:\n                head = TextHead(channels).to(device)\n                head.load_state_dict(blob["text_head"])\n                head.eval()\n                print(f"Loaded text head: {candidate}")\n                return head\n        print("No saved text head found; the text-head column will be blank.")\n        return None\n\n    return {\n        "torch": torch,\n        "F": F,\n        "device": device,\n        "test_ds": test_ds,\n        "test_loader": test_loader,\n        "build_model": build_model,\n        "wire_prior": wire_prior,\n        "predict": predict,\n        "capture_row": capture_row,\n        "load_text_head": load_text_head,\n    }\n\n\ndef component_stats(mask: np.ndarray) -> tuple[int, int]:\n    import cv2\n\n    n, _labels, stats, _centroids = cv2.connectedComponentsWithStats(mask.astype(np.uint8), 8)\n    areas = [int(a) for a in stats[1:, cv2.CC_STAT_AREA] if int(a) > 0]\n    return len(areas), min(areas) if areas else 0\n\n\ndef select_indices(args: argparse.Namespace, ctx: dict) -> tuple[list[int], list[dict]]:\n    torch = ctx["torch"]\n    test_ds = ctx["test_ds"]\n    build_model = ctx["build_model"]\n    wire_prior = ctx["wire_prior"]\n    predict = ctx["predict"]\n\n    if args.explicit_indices.strip():\n        indices = [int(x) for x in args.explicit_indices.split(",") if x.strip()]\n        return indices, [{"idx": i, "label": f"selected #{j + 1}"} for j, i in enumerate(indices)]\n\n    prior_path = Path(args.final_out) / "arms" / f"{args.prior_arm}.pth"\n    require(prior_path, "+prior checkpoint")\n\n    base_model = build_model()\n    base_model.eval()\n    prior_model = build_model()\n    wire_prior(prior_model)\n    state = torch.load(prior_path, map_location="cpu")["state_dict"]\n    prior_model.load_state_dict({k.replace("module.", ""): v for k, v in state.items()}, strict=False)\n    prior_model.eval()\n\n    rows = []\n    n_scan = min(len(test_ds), args.scan_limit) if args.scan_limit else len(test_ds)\n    for idx in range(n_scan):\n        sample = test_ds[idx]\n        gt = sample["label"][0].numpy() > 0\n        if not gt.any():\n            continue\n        p_base = predict(base_model, idx, False)\n        p_prior = predict(prior_model, idx, True)\n        diff = np.abs(p_prior - p_base)\n        changed = int(((p_base > args.threshold) != (p_prior > args.threshold)).sum())\n        n_comp, min_comp = component_stats(gt)\n        rows.append(\n            {\n                "idx": idx,\n                "p_base": p_base,\n                "p_prior": p_prior,\n                "maxdiff": float(diff.max()),\n                "meandiff": float(diff.mean()),\n                "changed": changed,\n                "n_comp": n_comp,\n                "min_comp": min_comp,\n                "area": int(gt.sum()),\n            }\n        )\n        if (idx + 1) % 25 == 0:\n            print(f"scanned {idx + 1}/{n_scan}")\n\n    chosen = sorted(rows, key=lambda r: (-r["maxdiff"], -r["changed"]))[: args.n_top_diff]\n    used = {r["idx"] for r in chosen}\n\n    medium = [\n        r\n        for r in rows\n        if r["idx"] not in used and r["n_comp"] == 1 and 200 <= r["min_comp"] <= 700 and r["changed"] > 0\n    ]\n    if medium:\n        chosen.append(sorted(medium, key=lambda r: (abs(r["min_comp"] - 365), -r["changed"]))[0])\n        used.add(chosen[-1]["idx"])\n\n    small = [\n        r\n        for r in rows\n        if r["idx"] not in used and r["n_comp"] <= 3 and 1 <= r["min_comp"] <= 30 and r["changed"] > 0\n    ]\n    if small:\n        chosen.append(sorted(small, key=lambda r: (abs(r["min_comp"] - 10), -r["changed"]))[0])\n\n    labels = []\n    for rank, row in enumerate(chosen):\n        if rank < args.n_top_diff:\n            name = f"max-diff #{rank + 1}"\n        elif row["n_comp"] == 1:\n            name = f"medium\\\\n{row[\'n_comp\']} comp min {row[\'min_comp\']}px"\n        else:\n            name = f"single-small\\\\n{row[\'n_comp\']} comps min {row[\'min_comp\']}px"\n        labels.append({**row, "label": name})\n\n    del base_model, prior_model\n    if ctx["device"] == "cuda":\n        torch.cuda.empty_cache()\n    return [r["idx"] for r in labels], labels\n\n\ndef overlay_contours(image: np.ndarray, gt=None, pred=None, pred_color=(255, 190, 0)) -> np.ndarray:\n    import cv2\n\n    out = image.copy()\n    if gt is not None:\n        contours, _ = cv2.findContours(gt.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)\n        cv2.drawContours(out, contours, -1, (0, 210, 255), 2)\n    if pred is not None:\n        contours, _ = cv2.findContours(pred.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)\n        cv2.drawContours(out, contours, -1, pred_color, 2)\n    return out\n\n\ndef save_figure(args: argparse.Namespace, ctx: dict, labels: list[dict]) -> None:\n    import matplotlib.pyplot as plt\n\n    build_model = ctx["build_model"]\n    wire_prior = ctx["wire_prior"]\n    predict = ctx["predict"]\n    capture_row = ctx["capture_row"]\n    load_text_head = ctx["load_text_head"]\n    torch = ctx["torch"]\n\n    prior_path = Path(args.final_out) / "arms" / f"{args.prior_arm}.pth"\n    require(prior_path, "+prior checkpoint")\n\n    base_model = build_model()\n    base_model.eval()\n    prior_model = build_model()\n    wire_prior(prior_model)\n    state = torch.load(prior_path, map_location="cpu")["state_dict"]\n    prior_model.load_state_dict({k.replace("module.", ""): v for k, v in state.items()}, strict=False)\n    prior_model.eval()\n    text_head = load_text_head()\n\n    n_rows = len(labels)\n    titles = [\n        "Document + GT",\n        "OCR mask M_text",\n        "Text head (frozen VPH)",\n        "P(ours + prior)",\n        "GT + baseline",\n        "GT + ours (+prior)",\n    ]\n    fig, axes = plt.subplots(n_rows, len(titles), figsize=(18.8, 3.0 * n_rows), constrained_layout=True)\n    axes = np.atleast_2d(axes)\n\n    for r, info in enumerate(labels):\n        row = capture_row(base_model, text_head, info["idx"])\n        if "p_base" in info and "p_prior" in info:\n            p_base = info["p_base"]\n            p_prior = info["p_prior"]\n        else:\n            p_base = predict(base_model, info["idx"], False)\n            p_prior = predict(prior_model, info["idx"], True)\n            diff = np.abs(p_prior - p_base)\n            info["maxdiff"] = float(diff.max())\n            info["changed"] = int(((p_base > args.threshold) != (p_prior > args.threshold)).sum())\n\n        pred_base = p_base > args.threshold\n        pred_prior = p_prior > args.threshold\n        panels = [\n            overlay_contours(row["image"], gt=row["gt"]),\n            row["ocr"],\n            row["text"] if row["text"] is not None else np.zeros_like(row["gt"], dtype=np.float32),\n            p_prior,\n            overlay_contours(row["image"], gt=row["gt"], pred=pred_base, pred_color=(220, 40, 40)),\n            overlay_contours(row["image"], gt=row["gt"], pred=pred_prior, pred_color=(255, 190, 0)),\n        ]\n        cmaps = [None, "viridis", "viridis", "magma", None, None]\n\n        for c, (panel, cmap) in enumerate(zip(panels, cmaps)):\n            ax = axes[r, c]\n            if cmap is None:\n                ax.imshow(panel)\n            else:\n                ax.imshow(panel, cmap=cmap, vmin=0, vmax=1)\n            ax.set_xticks([])\n            ax.set_yticks([])\n            if r == 0:\n                ax.set_title(titles[c], fontsize=10)\n\n        axes[r, 0].set_ylabel(info["label"], fontsize=8)\n        axes[r, 5].set_xlabel(\n            f"{int(info.get(\'changed\', 0)):,} px changed   max|d|={float(info.get(\'maxdiff\', 0.0)):.3f}",\n            fontsize=8,\n        )\n\n    out = Path(args.out)\n    out.parent.mkdir(parents=True, exist_ok=True)\n    fig.savefig(out, bbox_inches="tight")\n    fig.savefig(out.with_suffix(".png"), dpi=220, bbox_inches="tight")\n    print(f"saved {out}")\n    print(f"saved {out.with_suffix(\'.png\')}")\n\n\ndef main() -> None:\n    args = parse_args()\n    setup_runtime(args)\n    ctx = build_components(args)\n    _indices, labels = select_indices(args, ctx)\n    print("selected rows:")\n    for row in labels:\n        print(\n            f"  idx={row[\'idx\']:>4} {row[\'label\'].replace(chr(10), \' \'):<28} "\n            f"changed={int(row.get(\'changed\', 0)):>5} max|d|={float(row.get(\'maxdiff\', 0.0)):.3f}"\n        )\n    save_figure(args, ctx, labels)\n\n\nif __name__ == "__main__":\n    main()\n')
print('Wrote', script_path)


91:    doctamper_models = require(Path(args.doctamper_dir) / "models", "DocTamper/models")
92:    sys.path.insert(0, str(doctamper_models))
93:    sys.path.insert(0, str(project_dir))
94:    os.chdir(doctamper_models)


In [20]:
# Preflight check: these must exist before running the figure generator.
required = {
    'DTD checkpoint': CHECKPOINT_DIR / 'dtd_doctamper.pth',
    'VPH checkpoint': CHECKPOINT_DIR / 'vph_imagenet.pt',
    'Swin checkpoint': CHECKPOINT_DIR / 'swin_imagenet.pt',
    '+prior arm checkpoint': FINAL_OUT / 'arms' / f'{PRIOR_ARM}.pth',
    'test manifest': MANIFEST_DIR / 'test.json',
    'DocTamper models': DOCTAMPER_DIR / 'models',
}
missing = []
for name, path in required.items():
    ok = path.exists()
    print(f'{name:24s}', 'OK' if ok else 'MISSING', path)
    if not ok:
        missing.append((name, path))
assert not missing, 'Missing required files/folders. Fix the paths above or copy the missing Drive outputs.'


DTD checkpoint           OK /content/drive/MyDrive/HLCV/checkpoints/dtd_doctamper.pth
VPH checkpoint           OK /content/drive/MyDrive/HLCV/checkpoints/vph_imagenet.pt
Swin checkpoint          OK /content/drive/MyDrive/HLCV/checkpoints/swin_imagenet.pt
+prior arm checkpoint    OK /content/drive/MyDrive/HLCV_samira/final_seed42/arms/easyocr.pth
test manifest            OK /content/drive/MyDrive/HLCV_samira/manifests/train800_val200_test200_seed42/test.json
DocTamper models         OK /content/DocTamper/models


In [14]:
# Generate the PDF.
%cd /content/HLCV-Project
!PYTHONPATH=/content/DocTamper/models:/content/HLCV-Project:$PYTHONPATH \
python scripts/make_qualitative_with_probability_heatmap.py \
  --project-dir "$PROJECT_DIR" \
  --doctamper-dir "$DOCTAMPER_DIR" \
  --data-root "$DATA_ROOT" \
  --manifest-dir "$MANIFEST_DIR" \
  --checkpoint-dir "$CHECKPOINT_DIR" \
  --final-out "$FINAL_OUT" \
  --ta-out "$TA_OUT" \
  --prior-arm "$PRIOR_ARM" \
  --out "$OUT_PDF"


Wrote /content/HLCV-Project/scripts/make_qualitative_with_probability_heatmap.py


In [22]:
# Preview and download.
from IPython.display import Image, display
from google.colab import files

OUT_PNG = OUT_PDF.with_suffix('.png')
print('PDF:', OUT_PDF)
print('PNG:', OUT_PNG)
if OUT_PNG.exists():
    display(Image(filename=str(OUT_PNG)))
if OUT_PDF.exists():
    files.download(str(OUT_PDF))


PDF: /content/drive/MyDrive/HLCV_samira/qualitative/qualitative_with_probability_heatmap.pdf
PNG: /content/drive/MyDrive/HLCV_samira/qualitative/qualitative_with_probability_heatmap.png
